# ANSYS Journal File Editor

Create and edit ANSYS journal files (.jou) easily in this notebook. Write your journal commands, then export them directly as .jou files for use in ANSYS.

In [7]:
import os
from pathlib import Path
from datetime import datetime
import tempfile

# Configuration
DEFAULT_EXPORT_DIR = os.path.expanduser("~/Desktop")  # Change this to your preferred directory
CURRENT_PROJECT = "ANSYS_Journal"

# Create a sample template
SAMPLE_JOURNAL = """! ANSYS Journal File
! Created: {timestamp}
! Description: Your journal file description here

! Enter your ANSYS commands below

! Example commands:
! /PREP7
! ET,1,SOLID185
! BLOCK,0,10,0,10,0,10
! ESIZE,1
! VMESH,ALL
! /SOLU
! ANTYPE,STATIC
! SOLVE
! /POST1
"""

print("✓ ANSYS Journal File Editor loaded")
print(f"✓ Default export directory: {DEFAULT_EXPORT_DIR}")


✓ ANSYS Journal File Editor loaded
✓ Default export directory: C:\Users\<YOUR_USERNAME>/Desktop


## Configuration & Control Panel

Change all settings below before editing your journal file

In [8]:
# ============================================================
# CONFIGURATION - MODIFY THIS SECTION BEFORE EDITING YOUR JOURNAL
# ============================================================

# OUTPUT SETTINGS
export_filename = "4.3.1.4.G.0_5711"  # Change this to your desired filename (no .jou needed)
export_directory = r"C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Data_Prepartation\4.3.1.4.G_setup"  # Where to save the .jou file


# After editing, you can:
#   1. Run the EXPORT cell to save your changes
#   2. Run VALIDATE to check for syntax issues
#   3. Run LIST FILES to see all your saved journals

print("=" * 70)
print(f"📁 Export filename: {export_filename}.jou")
print(f"📁 Export directory: {export_directory}")

print("=" * 70)

📁 Export filename: 4.3.1.4.G.0_5711.jou
📁 Export directory: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Data_Prepartation\4.3.1.4.G_setup


In [9]:
# ============ EDIT YOUR JOURNAL CONFIGURATION HERE ============

# --- 1. Angle of Attack (AoA) Settings ---
# Choose Mode: "Range" (start, end, step) OR "List" (specific values)
AOA_MODE = "List"  # Options: "Range", "List"

# If "Range":
AOA_START = 0
AOA_END = 7
AOA_STEP = 1

# If "List":
AOA_LIST = [0,1,2,3,4,5,7,11]

# --- 2. Simulation Parameters ---
V_MAG = 24.38       # Velocity magnitude (m/s)
BASE_OUTPUT_DIR = "/home/ljcrick/directories/4.3.1.4.NG"
OUTPUT_FILENAME_BASE = "4.3.1.4.G" # Base name for output files (e.g., config name)
DRAG_REPORT_FILE = "drag-rfile"
LIFT_REPORT_FILE = "lift-rfile"
ITERATIONS = 1200   # Number of iterations to run
TEST_MODE = False    # If True: Updates BCs but skips Solving & Saving data. Use to verify setup.



# --- 3. Define Zone Groups ---
# Organize your boundaries into logical groups.

GROUP_INLET_MAIN = [
    "inlet-enclosure-enclosure_instance_29_solid1",
    "inlet-enclosure-enclosure_instance_28_solid1",
    "inlet-enclosure-enclosure_instance_47_solid1",
    "inlet-enclosure-enclosure_instance_26_solid1",
    "inlet-enclosure-enclosure_instance_38_solid1",
    "inlet-enclosure-enclosure_instance_12_solid1"
]

GROUP_INLET_BOTTOM = [
    "inlet_bottom-enclosure-enclosure_instance_10_solid1",
    "inlet_bottom-enclosure-enclosure_instance_11_solid1",
    "inlet_bottom-enclosure-enclosure_instance_12_solid1",
    "inlet_bottom-enclosure-enclosure_instance_4_solid1",
    "inlet_bottom-enclosure-enclosure_instance_5_solid1",
    "inlet_bottom-enclosure-enclosure_instance_6_solid1",
    "inlet_bottom-enclosure-enclosure_instance_7_solid1",
    "inlet_bottom-enclosure-enclosure_instance_8_solid1",
    "inlet_bottom-enclosure-enclosure_instance_9_solid1"
]

GROUP_OUTLET_MAIN = [
    "outlet-enclosure-enclosure_instance_18_solid1",
    "outlet-enclosure-enclosure_instance_27_solid1",
    "outlet-enclosure-enclosure_instance_30_solid1",
    "outlet-enclosure-enclosure_instance_39_solid1",
    "outlet-enclosure-enclosure_instance_4_solid1"
]

GROUP_OUTLET_TOP = [
    "outlet_top-enclosure-enclosure_instance_39_solid1",
    "outlet_top-enclosure-enclosure_instance_40_solid1",
    "outlet_top-enclosure-enclosure_instance_41_solid1",
    "outlet_top-enclosure-enclosure_instance_42_solid1",
    "outlet_top-enclosure-enclosure_instance_43_solid1",
    "outlet_top-enclosure-enclosure_instance_44_solid1",
    "outlet_top-enclosure-enclosure_instance_45_solid1",
    "outlet_top-enclosure-enclosure_instance_46_solid1",
    "outlet_top-enclosure-enclosure_instance_47_solid1"
]

GROUP_SYMMETRY = []

# --- 4. Zone Logic Configuration ---

# DEFAULT CONFIGURATION (Apply to all AoAs unless overridden)
DEFAULT_ZONES = {
    "inlets":   GROUP_INLET_MAIN + GROUP_INLET_BOTTOM,
    "outlets":  GROUP_OUTLET_MAIN + GROUP_OUTLET_TOP,
    "symmetry": GROUP_SYMMETRY
}

# OVERRIDE RULES
# Add rules to change zones for specific AoAs. 
# You can use "aoa_list" (specific values) OR "aoa_range" (min, max).
# Rules are processed in order; later rules overwrite earlier ones.

ZONE_RULES = [
    # Example 1: Use specific zones for AoA 0, 2.5, and 4
    # {
    #     "aoa_list": [0, 2.5, 4],
    #     "inlets": GROUP_INLET_MAIN,   # Only use MAIN inlets
    #     "outlets": GROUP_OUTLET_MAIN, # Only use MAIN outlets
    #     "symmetry": ["sym_zone_1"]    # Specific symmetry zones
    # },
    
    # Example 2: Use specific zones for all AoAs between 0 and 4
     {
         "aoa_range": [0, 4],  # Start, End (Inclusive)
         "inlets": GROUP_INLET_MAIN, 
         "outlets": GROUP_OUTLET_MAIN + GROUP_OUTLET_TOP,
         "symmetry": GROUP_INLET_BOTTOM
     }
]

# TUI Command Strings
# HOW TO FIND THE CORRECT SEQUENCE:
# 1. Open Fluent TUI (Console).
# 2. Type: /define/boundary-conditions/pressure-outlet
# 3. Enter a zone name when prompted.
# 4. NOTE DOWN the order of questions asked. 
#    - If it asks "Gauge Pressure?", your string should start with "0" (or your value).
#    - If it asks "Radial Equilibrium?", start with "no".
#    - If it asks "yes/no", type "yes" or "no".

INLET_TUI_SETTINGS = "no yes yes no 0 yes no ~a no ~a no ~a no no yes 0.05 10"

# Updated based on user's TUI output (Jan 19, 2026)
OUTLET_TUI_SETTINGS = "yes no 0 no yes no no yes 5 10 yes no no no"


# ============================================================
# AUTOMATED JOURNAL GENERATION (DO NOT EDIT BELOW THIS LINE)
# ============================================================
import numpy as np

# 1. Determine Full AoA sequence
if AOA_MODE == "Range":
    aoa_values = list(np.arange(AOA_START, AOA_END + 0.0001, AOA_STEP))
    config_note = f"Range: {AOA_START} to {AOA_END} step {AOA_STEP}"
else:
    aoa_values = AOA_LIST
    config_note = f"List: {AOA_LIST}"

# Helper: Format lists for Scheme
def format_scheme_list(py_list):
    if not py_list: return "'()"
    quoted_items = [f'"{item}"' for item in py_list]
    return f"'({' '.join(quoted_items)})"

# 2. Start building the Journal Content
journal_content = f"""; ============================================================
; ANSYS Fluent Journal File — Auto-Generated
; Config: {config_note}
; Velocity: {V_MAG} m/s
; Test Mode: {TEST_MODE}
; ============================================================

(define base-output-dir "{BASE_OUTPUT_DIR}")
(define V_mag {V_MAG})

(define drag-report-file-name "{DRAG_REPORT_FILE}")
(define lift-report-file-name "{LIFT_REPORT_FILE}")

(define (deg-to-rad deg) (* deg (/ 3.14159265359 180.0)))
(define (ensure-directory dir-path)
  (system (format #f "mkdir \\"~a\\"" dir-path)))

; Create base directory
(ensure-directory base-output-dir)
"""

# 3. Iterate through each AoA in Python
if 'TUI_DEBUG_MODE' in globals() and TUI_DEBUG_MODE:
    print("⚠ TUI_DEBUG_MODE is ON: Limiting to first AoA and first Zone per group.")
    aoa_values = aoa_values[:1]

for aoa in aoa_values:
    # 1. Start with defaults
    target_inlets = DEFAULT_ZONES["inlets"]
    target_outlets = DEFAULT_ZONES["outlets"]
    target_symmetry = DEFAULT_ZONES["symmetry"]

    # 2. Apply Override Rules
    for rule in ZONE_RULES:
        is_match = False
        
        # Check explicit list
        if "aoa_list" in rule:
             if aoa in rule["aoa_list"]:
                 is_match = True
        
        # Check range [start, end] inclusive
        if not is_match and "aoa_range" in rule:
            r_start, r_end = rule["aoa_range"]
            if r_start <= aoa <= r_end:
                is_match = True
        
        # Apply strict overrides if match found
        if is_match:
            if "inlets" in rule: target_inlets = rule["inlets"]
            if "outlets" in rule: target_outlets = rule["outlets"]
            if "symmetry" in rule: target_symmetry = rule["symmetry"]
    
    # DEBUG TRIMMING
    if 'TUI_DEBUG_MODE' in globals() and TUI_DEBUG_MODE:
        target_inlets = target_inlets[:1] if target_inlets else []
        target_outlets = target_outlets[:1] if target_outlets else []
        target_symmetry = target_symmetry[:1] if target_symmetry else []

    # 3. Convert to Scheme strings
    s_inlets = format_scheme_list(target_inlets)
    s_outlets = format_scheme_list(target_outlets)
    s_symmetries = format_scheme_list(target_symmetry)
    
    # 4. Handle Test Mode Logic
    if TEST_MODE:
        solve_block = '; [TEST MODE] Solver skipped'
        save_block = '; [TEST MODE] Save skipped'
        iter_display = f'(display "  [TEST] Applying BCs for AoA {aoa}, skipping solve/save...\\n")'
    else:
        solve_block = f'(ti-menu-load-string "/solve/iterate {ITERATIONS}")'
        save_block = '(ti-menu-load-string (format #f "/file/write-case-data ~a/~a yes" current-aoa-dir case-data-name))'
        iter_display = f'(display "Running {ITERATIONS} iterations for AoA {aoa}...\\n")'

    journal_content += f"""
; ------------------------------------------------------------
; Running AoA = {aoa}
; ------------------------------------------------------------
(define aoa {aoa})
(define current-aoa-dir (format #f "~a/AoA_~a" base-output-dir aoa))
(ensure-directory current-aoa-dir)

(display (format #f "~%===== Running AoA = ~a =====~%" aoa))

; Calculate Velocity Components
(define aoa_rad (deg-to-rad aoa))
(define V_x (* V_mag (cos aoa_rad)))
(define V_y (* V_mag (sin aoa_rad)))
(define V_z 0.0)

; Update Report Files
(define new-drag-path (format #f "~a/drag_force_AoA_~a.txt" current-aoa-dir aoa))
(define new-lift-path (format #f "~a/lift_force_AoA_~a.txt" current-aoa-dir aoa))

(ti-menu-load-string (format #f "/solve/report-files/edit ~a file-name \\"~a\\" q" drag-report-file-name new-drag-path))
(ti-menu-load-string (format #f "/solve/report-files/edit ~a file-name \\"~a\\" q" lift-report-file-name new-lift-path))

; Apply Boundary Conditions
(define inlet-list {s_inlets})
(define outlet-list {s_outlets})
(define symmetry-list {s_symmetries})

; Inlets
(if (not (null? inlet-list))
  (for-each
    (lambda (zone)
      ; FORCE ZONE TYPE TO VELOCITY-INLET FIRST
      ; This prevents "Invalid zone" errors if the zone was previously Symmetry
      (ti-menu-load-string (format #f "/define/boundary-conditions/zone-type ~a velocity-inlet" zone))
      
      ; Now apply settings
      (ti-menu-load-string
        (format #f "/define/boundary-conditions/velocity-inlet ~a {INLET_TUI_SETTINGS}" zone V_x V_y V_z)))
    inlet-list))

; Outlets
(if (not (null? outlet-list))
  (for-each
    (lambda (zone)
        (ti-menu-load-string (format #f "/define/boundary-conditions/zone-type ~a pressure-outlet" zone))
        (ti-menu-load-string (format #f "/define/boundary-conditions/pressure-outlet ~a {OUTLET_TUI_SETTINGS}" zone)))
    outlet-list))

; Symmetry
(if (not (null? symmetry-list))
  (for-each
    (lambda (zone)
        (ti-menu-load-string (format #f "/define/boundary-conditions/zone-type ~a symmetry" zone)))
    symmetry-list))

; Solve
{iter_display}
{solve_block}

; Save
(define case-data-name (format #f "{OUTPUT_FILENAME_BASE}.~a.cas.h5" aoa))
{save_block}
"""

journal_content += '\n(display "~%=== AoA sweep completed successfully ===~%")\n'

print(f"✓ Journal content generated for {len(aoa_values)} AoA steps")
if 'TUI_DEBUG_MODE' in globals() and TUI_DEBUG_MODE:
    print("⚠ DEBUG MODE ACTIVE: Generated journal for only 1 AoA and 1 Zone.")
if TEST_MODE:
    print("⚠ TEST MODE: Solving and saving disabled.")


✓ Journal content generated for 8 AoA steps


In [10]:
print(journal_content)

; ============================================================
; ANSYS Fluent Journal File — Auto-Generated
; Config: List: [0, 1, 2, 3, 4, 5, 7, 11]
; Velocity: 24.38 m/s
; Test Mode: False
; ============================================================

(define base-output-dir "/home/ljcrick/directories/4.3.1.4.NG")
(define V_mag 24.38)

(define drag-report-file-name "drag-rfile")
(define lift-report-file-name "lift-rfile")

(define (deg-to-rad deg) (* deg (/ 3.14159265359 180.0)))
(define (ensure-directory dir-path)
  (system (format #f "mkdir \"~a\"" dir-path)))

; Create base directory
(ensure-directory base-output-dir)

; ------------------------------------------------------------
; Running AoA = 0
; ------------------------------------------------------------
(define aoa 0)
(define current-aoa-dir (format #f "~a/AoA_~a" base-output-dir aoa))
(ensure-directory current-aoa-dir)

(display (format #f "~%===== Running AoA = ~a =====~%" aoa))

; Calculate Velocity Components
(define a

## Export Journal File

Use the functions below to save your journal file as .jou

In [11]:
def export_journal(filename, content, directory=None):
    """
    Export journal content to a .jou file
    
    Parameters:
    -----------
    filename : str
        Name of the file (with or without .jou extension)
    content : str
        The journal content to export
    directory : str, optional
        Directory to save the file. If None, uses DEFAULT_EXPORT_DIR
    
    Returns:
    --------
    str : Full path to the exported file
    """
    if directory is None:
        directory = DEFAULT_EXPORT_DIR
    
    # Ensure directory exists
    os.makedirs(directory, exist_ok=True)
    
    # Add .jou extension if not present
    if not filename.lower().endswith('.jou'):
        filename = filename + '.jou'
    
    # Create full file path
    filepath = os.path.join(directory, filename)
    
    # Write content to file with UTF-8 encoding
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
    
    return filepath


def export_current_journal(filename, directory=None):
    """
    Export the current journal_content to a .jou file
    
    Parameters:
    -----------
    filename : str
        Name of the file (with or without .jou extension)
    directory : str, optional
        Directory to save the file
    """
    filepath = export_journal(filename, journal_content, directory)
    
    # Get file size
    file_size = os.path.getsize(filepath)
    
    print(f"✓ Journal file exported successfully!")
    print(f"  Filename: {os.path.basename(filepath)}")
    print(f"  Location: {filepath}")
    print(f"  Size: {file_size} bytes")
    print(f"\n✓ Ready to use in ANSYS!")
    
    return filepath


# Example usage - uncomment and modify the line below to export:
# export_current_journal("my_simulation")

print("✓ Export functions ready")
print("✓ To export your journal, run the next cell")

✓ Export functions ready
✓ To export your journal, run the next cell


In [12]:
# ============================================================
# QUICK ACTIONS - RUN THESE AFTER EDITING YOUR JOURNAL
# ============================================================

# EXPORT YOUR JOURNAL
print("\n" + "="*70)
print("EXPORTING JOURNAL FILE...")
print("="*70)
filepath = export_current_journal(export_filename, export_directory)

# Display quick reference
print("\n✓ YOUR JOURNAL IS READY!")
print(f"  File: {filepath}")
print(f"  You can now use this in ANSYS!")



EXPORTING JOURNAL FILE...
✓ Journal file exported successfully!
  Filename: 4.3.1.4.G.0_5711.jou
  Location: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Data_Prepartation\4.3.1.4.G_setup\4.3.1.4.G.0_5711.jou
  Size: 31687 bytes

✓ Ready to use in ANSYS!

✓ YOUR JOURNAL IS READY!
  File: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Data_Prepartation\4.3.1.4.G_setup\4.3.1.4.G.0_5711.jou
  You can now use this in ANSYS!


## Quick Reference - Load Files & Utilities (Optional)

In [ ]:
def load_journal(filepath):
    """
    Load an existing journal file
    
    Parameters:
    -----------
    filepath : str
        Path to the journal file
    
    Returns:
    --------
    str : Content of the journal file
    """
    if not os.path.exists(filepath):
        print(f"✗ File not found: {filepath}")
        return None
    
    with open(filepath, 'r') as f:
        content = f.read()
    
    file_size = os.path.getsize(filepath)
    lines = len(content.split('\n'))
    
    print(f"✓ Journal file loaded successfully!")
    print(f"  Filename: {os.path.basename(filepath)}")
    print(f"  Size: {file_size} bytes")
    print(f"  Lines: {lines}")
    
    return content


def list_journal_files(directory=None):
    """
    List all .jou files in a directory
    
    Parameters:
    -----------
    directory : str, optional
        Directory to search. If None, uses DEFAULT_EXPORT_DIR
    """
    if directory is None:
        directory = DEFAULT_EXPORT_DIR
    
    if not os.path.exists(directory):
        print(f"✗ Directory not found: {directory}")
        return []
    
    jou_files = [f for f in os.listdir(directory) if f.lower().endswith('.jou')]
    
    if not jou_files:
        print(f"No .jou files found in {directory}")
        return []
    
    print(f"Found {len(jou_files)} journal file(s) in {directory}:")
    print("-" * 60)
    
    for i, filename in enumerate(jou_files, 1):
        filepath = os.path.join(directory, filename)
        size = os.path.getsize(filepath)
        mod_time = datetime.fromtimestamp(os.path.getmtime(filepath)).strftime("%Y-%m-%d %H:%M:%S")
        print(f"{i:2}. {filename:<40} {size:>10} bytes  {mod_time}")
    
    print("-" * 60)
    return jou_files


# ============================================================
# USE THESE HELPERS BELOW
# ============================================================

# TO LOAD AN EXISTING JOURNAL:
# Step 1: Find your file
# list_journal_files("C:/path/to/folder")
#
# Step 2: Load it
# my_journal = load_journal("C:/path/to/your/journal.jou")
#
# Step 3: Use it (copy the content and paste into cell 3)
# print(my_journal)

print("✓ Utility functions ready")


## Tips & Advanced Features

In [ ]:
def validate_journal(content):
    """
    Basic validation of journal syntax
    """
    lines = content.strip().split('\n')
    
    print(f"✓ Journal validation results:")
    print(f"  Total lines: {len(lines)}")
    print(f"  Non-comment lines: {sum(1 for l in lines if l.strip() and not l.strip().startswith('!'))}")
    print(f"  Comment lines: {sum(1 for l in lines if l.strip().startswith('!'))}")
    print(f"  Empty lines: {sum(1 for l in lines if not l.strip())}")
    
    return True


# ============================================================
# VALIDATE YOUR JOURNAL BEFORE EXPORTING
# ============================================================

# Run this to check your journal:
# validate_journal(journal_content)

print("✓ Advanced features ready")
